[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Why Test &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The three cells below rebuild what the notebook set up: the scratch folder on the import path,
Tuesday's readings as a list and as a file, `run_tests`, and the module and the summary script as
the notebook left them. Run them first, then the tasks in order, since task 6 reads the file task 5
writes. The last cell removes the scratch folder.


In [1]:
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
PROJECT.mkdir(parents=True, exist_ok=True)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
os.environ["NO_COLOR"] = "1"                  # programs started from here print errors without color codes

TUESDAY = ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
           "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]
(PROJECT / "tuesday.csv").write_text("\n".join(TUESDAY) + "\n", encoding="utf-8")


def run_tests(namespace):
    """Run every function in namespace whose name starts with test_, and report every failure."""
    tests = [value for name, value in namespace.items() if name.startswith("test_") and callable(value)]
    failed = 0
    for test in tests:
        try:
            test()
        except Exception as error:
            failed += 1
            print(f"FAILED {test.__name__}: {error!r}")
    print(f"{len(tests) - failed} passed, {failed} failed")


print("ready:", PROJECT)


ready: scratch/stations


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings."""
    by_station = {}
    for line in lines:
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


In [3]:
%%writefile scratch/stations/summary.py
"""Print each station's mean temperature from a file of readings: python summary.py readings.csv"""

import sys

from readings import summarize, to_fahrenheit

with open(sys.argv[1], encoding="utf-8") as file:
    for station, celsius in summarize(file).items():
        if celsius is None:
            print(station, "no readings")
        else:
            print(station, f"{celsius:.1f} C, {to_fahrenheit(celsius):.1f} F")


Writing scratch/stations/summary.py


In [4]:
import readings
importlib.reload(readings)                  # the file as it is now, if an older copy was imported

print(readings.summarize(TUESDAY))


{'Bergen': 5.0, 'Oslo': -2.0, 'Svalbard': None, 'Tromso': -6.0}


**1.** The mean of one reading.


In [5]:
def test_mean_of_one_reading():
    assert readings.mean([-6.3]) == -6.3


test_mean_of_one_reading()
print("test_mean_of_one_reading passed")


test_mean_of_one_reading passed


One reading is an edge worth a test: a mean that divided by the wrong count would get it wrong.


**2.** An empty reading, run on its own.


In [6]:
def test_an_empty_reading_is_none():
    assert readings.parse_reading("Svalbard,") == ("Svalbard", None)


run_tests({"test_an_empty_reading_is_none": test_an_empty_reading_is_none})


1 passed, 0 failed


The dictionary holds one test, so `run_tests` ran that test and nothing else, as passing `globals()`
would not have.


**3.** A test that fails on purpose.


In [7]:
def test_mean_is_the_larger_reading():
    assert readings.mean([4.2, 5.8]) == 5.8


run_tests({"test_mean_is_the_larger_reading": test_mean_is_the_larger_reading})


FAILED test_mean_is_the_larger_reading: AssertionError()
0 passed, 1 failed


The report names the test and the exception, `AssertionError()`, with nothing inside the
parentheses: it says that the check failed, and not what the mean was. pytest, in the **Your First
Test** notebook, shows the value.


**4.** A line that must raise.


In [8]:
def test_a_line_with_no_comma_raises_value_error():
    try:
        readings.parse_reading("Bergen")
    except ValueError:
        pass
    else:
        raise AssertionError("parse_reading('Bergen') raised nothing")


run_tests({"test_a_line_with_no_comma_raises_value_error": test_a_line_with_no_comma_raises_value_error})


1 passed, 0 failed


Without the `else`, the test would also pass if `parse_reading` raised nothing at all. The **Testing
Failure** notebook replaces the `try`, `except` and `else` with `pytest.raises`.


**5.** An integration test of a file.


In [9]:
(PROJECT / "wednesday.csv").write_text("Oslo,-1.8\nOslo,-2.2\nSvalbard,\n", encoding="utf-8")


def test_summary_of_wednesday():
    with open(PROJECT / "wednesday.csv", encoding="utf-8") as file:
        assert readings.summarize(file) == {"Oslo": -2.0, "Svalbard": None}


run_tests({"test_summary_of_wednesday": test_summary_of_wednesday})


1 passed, 0 failed


The test reads a real file, so it checks `summarize` together with the way a file hands it lines,
line endings and all.


**6.** An end-to-end test of the script.


In [10]:
def test_the_script_prints_wednesday():
    finished = subprocess.run([sys.executable, "summary.py", "wednesday.csv"], cwd=PROJECT, capture_output=True, text=True)
    assert finished.returncode == 0, finished.stderr.strip().splitlines()[-1]
    assert finished.stdout.splitlines() == ["Oslo -2.0 C, 28.4 F", "Svalbard no readings"]


run_tests({"test_the_script_prints_wednesday": test_the_script_prints_wednesday})


1 passed, 0 failed


The expected lines follow from the readings: the mean of `-1.8` and `-2.2` is `-2.0`, which is
`28.4` °F, and Svalbard sent nothing.

Last, remove the scratch folder:


In [11]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Why Test](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/01-why-test.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
